# M3L3 E08 — Router condicional con LangGraph (Resolution)
### Módulo 3 · Lecture 3 · Sistemas Multiagente

**Ejercicio paralelo:** E01 (clasificador de intenciones) + E03 (orquestador básico)

## ¿Qué vas a aprender hoy?
- usar el LLM para clasificar la intención del usuario dentro de un nodo.
- declarar routing condicional con `add_conditional_edges`.
- comparar el `if/else` manual de E03 con el routing declarativo de LangGraph.


## ¿Qué necesitás saber antes?

Venís de E07 donde aprendiste las cinco piezas de un grafo. En E08 agregamos la pieza que faltaba: el routing condicional, ahora con un LLM real haciendo la clasificación.

> **add_conditional_edges:** conecta un nodo a múltiples destinos posibles según el valor que devuelve una función de routing. El grafo decide en runtime cuál camino tomar.

En E01 y E03 el routing era un `if/else` con keywords. Ahora se usa el LLM para clasificar y el grafo para enrutar:

| E03 Python puro | E08 LangGraph |
|---|---|
| Keywords en `if/else` para clasificar | LLM clasifica con comprensión de lenguaje natural |
| `route(intent)` devuelve una función | `router(state)` devuelve un string con el nombre del nodo |
| El flujo está en el código de `handle_query` | El flujo está declarado en el grafo, visible en el diagrama |


## Paso 1 — Elegí tu proveedor de LLM

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

In [ ]:
!pip install langgraph -q

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph listo.")

## Sección 1 — El flujo del router

> **Router condicional:** nodo que clasifica la consulta usando el LLM y devuelve el nombre del nodo al que debe ir el flujo.

```
START
  |
  v
classify  --> (intent=hr)      --> hr_node      --> END
          --> (intent=tech)    --> tech_node    --> END
          --> (intent=billing) --> billing_node --> END
          --> (intent=unknown) --> unknown_node --> END
```

Reutilizamos la misma knowledge base de E03.

In [ ]:
knowledge_base = {
    "hr": [
        "Vacaciones: cada empleado tiene 15 días hábiles por año.",
        "Beneficios: el seguro médico inicia el primer día de trabajo.",
        "Licencias: registrar el pedido en PeopleOps y avisar al manager.",
    ],
    "tech": [
        "VPN: reiniciar el cliente, validar MFA y abrir ticket si persiste.",
        "Contraseña: restablecer desde el portal de identidad.",
        "Notebook: reportar equipo dañado con número de serie.",
    ],
    "billing": [
        "Facturas: cargar comprobantes antes del día 25.",
        "Reembolsos: adjuntar recibo, monto y centro de costo.",
        "Pagos: se procesan los viernes por la tarde.",
    ],
}

def retrieve(domain: str, query: str, k: int = 2) -> list:
    docs = knowledge_base[domain]
    words = set(query.lower().split())
    return sorted(docs, key=lambda d: sum(1 for w in words if w in d.lower()), reverse=True)[:k]

## Sección 2 — State y nodos

> **State del router:** guarda la consulta, el intent clasificado por el LLM y la respuesta final.

```
RouterState
  query    ← entrada del usuario
  intent   ← escrito por classify (LLM)
  response ← escrito por el nodo especialista
```

La función `router` solo lee `state["intent"]` y devuelve el nombre del nodo. Nunca ejecuta lógica de negocio.

In [ ]:
class RouterState(TypedDict):
    query: str
    intent: str
    response: str

In [ ]:
def classify(state: RouterState) -> dict:
    prompt = (
        "Clasificá esta consulta de soporte corporativo en exactamente una categoría.\n"
        "Categorías válidas: hr, tech, billing, unknown\n"
        "Respondé SOLO con la categoría, sin explicación ni puntuación.\n\n"
        f"Consulta: {state['query']}"
    )
    response = llm.invoke(prompt)
    intent = response.content.strip().lower()
    if intent not in ("hr", "tech", "billing"):
        intent = "unknown"
    return {"intent": intent}


def hr_node(state: RouterState) -> dict:
    docs = retrieve("hr", state["query"])
    return {"response": "HRAgent: " + " | ".join(docs)}


def tech_node(state: RouterState) -> dict:
    docs = retrieve("tech", state["query"])
    return {"response": "TechAgent: " + " | ".join(docs)}


def billing_node(state: RouterState) -> dict:
    docs = retrieve("billing", state["query"])
    return {"response": "BillingAgent: " + " | ".join(docs)}


def unknown_node(state: RouterState) -> dict:
    return {"response": "No puedo responder esa consulta."}


def router(state: RouterState) -> str:
    return {"hr": "hr_node", "tech": "tech_node", "billing": "billing_node"}.get(
        state["intent"], "unknown_node"
    )

## Sección 3 — Construir el grafo con routing condicional

> **add_conditional_edges(origen, fn_router, mapa):** conecta el nodo `origen` a distintos destinos según el string que devuelve `fn_router`. El `mapa` valida los valores posibles.

```python
graph.add_conditional_edges(
    "nodo_origen",
    funcion_router,
    {"valor_a": "nodo_a", "valor_b": "nodo_b"}
)
```

In [ ]:
graph = StateGraph(RouterState)

graph.add_node("classify",     classify)
graph.add_node("hr_node",      hr_node)
graph.add_node("tech_node",    tech_node)
graph.add_node("billing_node", billing_node)
graph.add_node("unknown_node", unknown_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges(
    "classify", router,
    {"hr_node": "hr_node", "tech_node": "tech_node",
     "billing_node": "billing_node", "unknown_node": "unknown_node"},
)
for node in ["hr_node", "tech_node", "billing_node", "unknown_node"]:
    graph.add_edge(node, END)

app = graph.compile()
print("Grafo compilado.")

El LLM clasifica con comprensión semántica — puede enrutar frases que no contienen las keywords exactas.

In [ ]:
for q in ["¿Cuántas vacaciones tengo?", "no me funciona la VPN", "cuándo se procesa mi factura", "qué hay para almorzar"]:
    r = app.invoke({"query": q, "intent": "", "response": ""})
    print(f"[{r['intent']:8s}] {r['response'][:60]}")

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    casos = [
        ({"query": "quiero pedir vacaciones",      "intent": "", "response": ""}, "hr",      "HRAgent"),
        ({"query": "no puedo conectarme a la VPN", "intent": "", "response": ""}, "tech",    "TechAgent"),
        ({"query": "necesito cargar una factura",  "intent": "", "response": ""}, "billing", "BillingAgent"),
        ({"query": "¿qué temperatura hace afuera?","intent": "", "response": ""}, "unknown", "No puedo"),
    ]
    for estado, intent_esp, prefix_esp in casos:
        r = app.invoke(estado)
        assert r["intent"] == intent_esp,   f"intent incorrecto para '{estado['query']}': {r['intent']}"
        assert prefix_esp in r["response"], f"respuesta incorrecta: {r['response']}"
    print("Checks E08 OK")

run_checks()

## ¿Qué aprendiste hoy?

- El LLM como clasificador entiende el lenguaje natural: puede enrutar consultas que nunca viste durante el diseño, sin necesitar una lista de keywords.
- `add_conditional_edges` separa la decisión de routing de la ejecución.
- El mapa de destinos válidos actúa como contrato: si `router` devuelve un valor no listado, LangGraph lanza error en compilación.

## Próximo ejercicio

En **E09** el LLM detecta múltiples dominios y la `Send` API despacha agentes en paralelo.
